# 11 — LightGBM: Histogram Boosting at Scale

**Difficulty:** ⭐⭐⭐⭐⭐ &nbsp;|&nbsp; **Estimated time:** 3 hours &nbsp;|&nbsp; **Topic:** Tree-Based Methods & Gradient Boosting

---

## Why This Matters

LightGBM (Light Gradient Boosting Machine, Ke et al. 2017) is one of the most widely used algorithms in competitive machine learning and production ML systems. It achieves **10–20× speedups** over traditional gradient boosting while matching or exceeding accuracy. If you work with tabular data at any scale, LightGBM is a first-line tool.

Three innovations make it fast:
1. **Histogram-based splitting** — bins continuous features, reduces split candidates from O(n) to O(255)
2. **GOSS** (Gradient-based One-Side Sampling) — discards easy training samples, focuses on hard ones
3. **Leaf-wise (best-first) tree growth** — always splits the most promising leaf, not all leaves at a depth

---

## Real-World Analogy First

Imagine you are a city planner deciding where to build a new school. A **traditional** planner visits every single house in the city (all 1 million of them), records its distance to the nearest school, and then finds the worst-served house.

A **LightGBM** planner works differently in three ways:

1. **Histogram trick:** Instead of examining each of the 1 million exact addresses, the planner first sorts all houses into 255 neighbourhoods (bins). Now they only check 255 neighbourhood representatives instead of 1 million houses — **roughly 4000× fewer evaluations**.

2. **GOSS trick:** The planner also notices that 80 % of houses are already well-served (small gradient = small error). They ignore most of those easy cases and focus attention on the 20 % that are poorly served (large gradient = large error). This makes each planning round much faster with little loss in quality.

3. **Leaf-wise trick:** Instead of simultaneously evaluating every zone at the same depth, the planner always digs deeper into **whichever zone shows the biggest improvement opportunity**. They might go 6 levels deep in the city centre while stopping at 2 levels in the countryside — spending effort where it counts.

---

## Learning Objectives

By the end of this notebook you will be able to:
- Explain histogram-based splitting and why it speeds up training
- Describe GOSS and why focusing on high-gradient samples is theoretically justified
- Contrast leaf-wise vs level-wise tree growth
- Configure key LightGBM hyperparameters (`num_leaves`, `min_data_in_leaf`, `feature_fraction`)
- Use LightGBM's native categorical feature support
- Benchmark LightGBM against XGBoost and sklearn GradientBoosting

## Section 1 — Setup & Synthetic House Price Dataset

We build a synthetic dataset of **1500 houses** with 10 features including 2 categorical features. This mirrors common real-estate ML tasks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.ensemble import GradientBoostingRegressor

np.random.seed(42)
sns.set_theme(style='whitegrid')

print('Libraries loaded.')

In [ ]:
# ── Install / import LightGBM with graceful fallback ──────────────────────────
try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
    print(f'LightGBM {lgb.__version__} available.')
except ImportError:
    LGBM_AVAILABLE = False
    print('LightGBM not installed. Falling back to sklearn GradientBoostingRegressor.')
    print('To install: pip install lightgbm')
    print('\nAll code cells will still run; notes explain what LightGBM does differently.')

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print(f'XGBoost {xgb.__version__} available.')
except ImportError:
    XGB_AVAILABLE = False
    print('XGBoost not installed — comparison section will use sklearn GB.')

In [ ]:
# ── Generate synthetic house price dataset ────────────────────────────────────
np.random.seed(42)
N = 1500

neighborhoods   = ['Downtown', 'Midtown', 'Uptown', 'Suburbs', 'Rural']
house_types     = ['Detached', 'Semi', 'Terrace', 'Flat']

neighborhood    = np.random.choice(neighborhoods, N)
house_type      = np.random.choice(house_types,   N)

sqft            = np.random.normal(1800, 500, N).clip(600, 4000)
bedrooms        = np.random.randint(1, 7, N).astype(float)
bathrooms       = np.random.randint(1, 5, N).astype(float)
age_years       = np.random.randint(0, 80, N).astype(float)
lot_size        = np.random.normal(5000, 2000, N).clip(1000, 15000)
garage_spaces   = np.random.randint(0, 4, N).astype(float)
dist_downtown   = np.random.exponential(5, N).clip(0.5, 30)
school_rating   = np.random.uniform(3, 10, N)

# Neighbourhood price premium
nbhd_premium    = {'Downtown': 60000, 'Midtown': 30000, 'Uptown': 45000,
                   'Suburbs': 10000, 'Rural': -20000}
type_premium    = {'Detached': 40000, 'Semi': 10000, 'Terrace': 0, 'Flat': -15000}

price = (
    120 * sqft
    + 15000 * bedrooms
    + 12000 * bathrooms
    - 800   * age_years
    + 8     * lot_size
    + 10000 * garage_spaces
    - 4000  * dist_downtown
    + 8000  * school_rating
    + np.array([nbhd_premium[n] for n in neighborhood])
    + np.array([type_premium[t] for t in house_type])
    + np.random.normal(0, 30000, N)          # noise
).clip(50000, 1200000)

df = pd.DataFrame({
    'sqft':          sqft,
    'bedrooms':      bedrooms,
    'bathrooms':     bathrooms,
    'age_years':     age_years,
    'lot_size':      lot_size,
    'garage_spaces': garage_spaces,
    'dist_downtown': dist_downtown,
    'school_rating': school_rating,
    'neighborhood':  neighborhood,
    'house_type':    house_type,
    'price':         price
})

print(f'Dataset shape: {df.shape}')
print(f'Price range:   ${df.price.min():,.0f} – ${df.price.max():,.0f}')
df.head()

In [ ]:
# ── Train / validation / test split ──────────────────────────────────────────
X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print(f'Train : {X_train.shape[0]} samples')
print(f'Val   : {X_val.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')

# Encoded copies for sklearn (replaces categoricals with integers)
cat_cols = ['neighborhood', 'house_type']
enc = OrdinalEncoder()
X_train_enc = X_train.copy()
X_val_enc   = X_val.copy()
X_test_enc  = X_test.copy()
X_train_enc[cat_cols] = enc.fit_transform(X_train[cat_cols])
X_val_enc[cat_cols]   = enc.transform(X_val[cat_cols])
X_test_enc[cat_cols]  = enc.transform(X_test[cat_cols])

## Section 2 — Innovation 1: Histogram-Based Splitting

### Plain English

Traditional gradient boosting finds the best split by sorting all values of a feature and scanning every possible threshold. For a feature with 1200 training samples, that means checking up to 1200 candidate splits — for every feature, at every node, for every tree.

LightGBM's histogram approach:
1. **Before training:** bin each continuous feature into at most 255 buckets (bins). Store bin index as `uint8` instead of `float64`.
2. **During training:** instead of scanning 1200 values, scan 255 bin boundaries.
3. **Benefit:** roughly 5–20× fewer split evaluations + 8× memory reduction (uint8 vs float64).

### Formula

For bin `b`, the histogram stores:
```
H[b].gradient_sum = Σ gᵢ  for all xᵢ in bin b
H[b].hessian_sum  = Σ hᵢ  for all xᵢ in bin b
H[b].count        = number of samples in bin b
```
The gain for a split at bin boundary `k` is then:
```
Gain(k) = ½ [ (ΣG_L)² / (ΣH_L + λ)  +  (ΣG_R)² / (ΣH_R + λ)  −  (ΣG)² / (ΣH + λ) ]  − γ
```
where G, H are summed over all bins to the left (L) or right (R) of the split point. **Subtraction trick:** compute the right side as `Total − Left` — only O(255) additions needed.

In [ ]:
# ── Histogram visualisation for 'sqft' ───────────────────────────────────────
feature = 'sqft'
values  = X_train[feature].values
n_bins  = 255

# Build histogram bins (as LightGBM would)
bin_edges  = np.linspace(values.min(), values.max(), n_bins + 1)
bin_labels = np.digitize(values, bin_edges) - 1
bin_labels = bin_labels.clip(0, n_bins - 1)

# Count samples per bin
counts = np.bincount(bin_labels, minlength=n_bins)
bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: raw values (simulating traditional sort)
axes[0].hist(values, bins=80, color='steelblue', alpha=0.7, edgecolor='white', linewidth=0.3)
axes[0].set_title('Traditional GB: evaluate all unique thresholds\n'
                  f'(up to {len(np.unique(values))} candidates)', fontsize=11)
axes[0].set_xlabel('sqft (raw)')
axes[0].set_ylabel('Count')
axes[0].axvline(np.median(values), color='red', linestyle='--', label='example split threshold')
axes[0].legend()

# Right: histogram bins
axes[1].bar(bin_centres, counts, width=(bin_edges[1]-bin_edges[0])*0.9,
            color='darkorange', alpha=0.8, edgecolor='white', linewidth=0.2)
axes[1].set_title(f'LightGBM: evaluate {n_bins} bin boundaries only\n'
                  f'(~{len(np.unique(values)) // n_bins}× fewer evaluations)', fontsize=11)
axes[1].set_xlabel('sqft (binned into 255 buckets)')
axes[1].set_ylabel('Bin count')
# Mark a bin boundary split
split_bin = 127
axes[1].axvline(bin_edges[split_bin], color='red', linestyle='--', label=f'bin boundary {split_bin}')
axes[1].legend()

plt.suptitle('Histogram-Based Splitting: Fewer Candidates, Same Quality', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'\nMemory comparison:')
print(f'  float64 storage: {values.nbytes / 1024:.1f} KB')
print(f'  uint8   storage: {bin_labels.astype(np.uint8).nbytes / 1024:.1f} KB  (8× smaller)')
print(f'  Unique values (original): {len(np.unique(values))}')
print(f'  Bin boundaries to check:  {n_bins}')

## Section 3 — Innovation 2: GOSS (Gradient-Based One-Side Sampling)

### Plain English

After a few boosting rounds, most training samples have small residuals — the model already predicts them well. These **low-gradient samples** contribute little to finding the next split. GOSS:

1. Sort samples by absolute gradient magnitude
2. Keep **all** top-`a` fraction (high-gradient = hard samples)
3. Randomly keep `b` fraction of the remaining (low-gradient = easy samples)
4. Multiply the kept low-gradient samples' gradients by `(1 − a) / b` to correct for the sampling bias

Typical values: `a = 0.2`, `b = 0.1` — means only 28 % of data is used per round, with ~3.6× speedup.

### Why This Is Theoretically Sound

The gain formula for a split depends on the sum of gradients in each child. By amplifying low-gradient samples by `(1−a)/b`, GOSS ensures that the estimated gradient sums remain **unbiased** — the split finder sees an unbiased estimate of the true gain even with fewer samples.

In [ ]:
# ── GOSS illustration ─────────────────────────────────────────────────────────
# Simulate gradients after a few boosting rounds
np.random.seed(42)
n_samples  = len(X_train)

# Gradients are typically right-skewed: most near 0, some large
gradients  = np.abs(np.random.exponential(0.3, n_samples))   # |gradient|

a = 0.20   # keep top-a fraction (high gradient)
b = 0.10   # sample b fraction of rest (low gradient)

sorted_idx     = np.argsort(-gradients)           # descending
n_high         = int(n_samples * a)
high_idx       = sorted_idx[:n_high]
low_idx        = sorted_idx[n_high:]
n_low_sampled  = int(len(low_idx) * b)
low_sampled    = np.random.choice(low_idx, n_low_sampled, replace=False)

category = np.full(n_samples, 'discarded (low gradient)', dtype=object)
category[low_sampled] = 'sampled low-gradient'
category[high_idx]    = 'kept high-gradient'

threshold = gradients[sorted_idx[n_high]]   # gradient threshold at a%

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: gradient distribution coloured by GOSS decision
colors = {'kept high-gradient': '#d62728', 'sampled low-gradient': '#1f77b4',
          'discarded (low gradient)': '#cccccc'}
for label, color in colors.items():
    mask = category == label
    axes[0].scatter(np.where(mask)[0], gradients[mask], c=color, s=4, alpha=0.5, label=label)
axes[0].axhline(threshold, color='red', linestyle='--', linewidth=1.5,
                label=f'gradient threshold ({threshold:.3f})')
axes[0].set_xlabel('Sample index (sorted by gradient)')
axes[0].set_ylabel('|Gradient|')
axes[0].set_title('GOSS: Sample Selection by Gradient Magnitude')
axes[0].legend(markerscale=2, fontsize=8)

# Right: pie chart showing data usage
total_used   = n_high + n_low_sampled
total_unused = n_samples - total_used
wedge_labels = [
    f'High-gradient kept\n({n_high} = {100*a:.0f}%)',
    f'Low-gradient sampled\n({n_low_sampled} = {100*n_low_sampled/n_samples:.1f}%)',
    f'Discarded\n({total_unused} = {100*total_unused/n_samples:.1f}%)'
]
wedge_sizes  = [n_high, n_low_sampled, total_unused]
wedge_colors = ['#d62728', '#1f77b4', '#cccccc']
axes[1].pie(wedge_sizes, labels=wedge_labels, colors=wedge_colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 9})
axes[1].set_title(f'GOSS Data Usage per Round\n'
                  f'Using {100*total_used/n_samples:.1f}% of samples per round')

plt.suptitle('Gradient-Based One-Side Sampling (GOSS)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'\nGOSS summary (a={a}, b={b}):')
print(f'  Total training samples : {n_samples}')
print(f'  High-gradient kept     : {n_high}   ({100*a:.0f}%)')
print(f'  Low-gradient sampled   : {n_low_sampled}    ({100*n_low_sampled/n_samples:.1f}%)')
print(f'  Used per round         : {total_used}  ({100*total_used/n_samples:.1f}%)')
print(f'  Amplification factor   : {(1-a)/b:.1f}×  (applied to low-gradient samples)')

## Section 4 — Innovation 3: Leaf-Wise vs Level-Wise Growth

### Plain English

**Level-wise (XGBoost, sklearn):** grow all leaves at depth `d` before moving to depth `d+1`. A complete binary tree with depth 5 has 32 leaves — you always get exactly 32 leaves.

**Leaf-wise (LightGBM):** at each step, find the single leaf whose split gives the **maximum gain** and split only that leaf. You might grow one branch to depth 10 while another stays at depth 2 — spending effort only where it reduces loss the most.

**Key trade-off:**
- Leaf-wise achieves lower training loss with fewer trees (or same number of leaves)
- But with small datasets or high `num_leaves`, it can overfit aggressively
- Control with: `num_leaves` (max total leaves) + `min_data_in_leaf` (min samples per leaf)

**Rule of thumb:** `num_leaves < 2^max_depth`  to avoid overfitting relative to a level-wise tree of the same depth.

In [ ]:
# ── Visualise leaf-wise vs level-wise growth ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

def draw_tree_node(ax, x, y, label, color, radius=0.35, fontsize=8):
    circle = plt.Circle((x, y), radius, color=color, zorder=3, linewidth=1.5,
                         edgecolor='black')
    ax.add_patch(circle)
    ax.text(x, y, label, ha='center', va='center', fontsize=fontsize,
            fontweight='bold', zorder=4)

def draw_edge(ax, x1, y1, x2, y2):
    ax.plot([x1, x2], [y1, y2], 'k-', linewidth=1.2, zorder=1)

# ── Level-wise (left) ────────────────────────────────────────────────────────
ax = axes[0]
ax.set_xlim(-1, 9)
ax.set_ylim(-1, 5.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Level-Wise Growth (XGBoost / sklearn)\n'
             'All nodes at depth d are split together', fontsize=10)

# Depth 0
draw_edge(ax, 4, 5, 2, 3.5); draw_edge(ax, 4, 5, 6, 3.5)
draw_tree_node(ax, 4, 5, 'Root\nd=0', '#AED6F1')
# Depth 1 — both split simultaneously (highlighted)
for xpos, lbl in [(2, 'L1\nd=1'), (6, 'R1\nd=1')]:
    draw_tree_node(ax, xpos, 3.5, lbl, '#F9E79F')  # yellow = 'being split'
# Depth 1 → 2 edges
draw_edge(ax, 2, 3.5, 1, 2); draw_edge(ax, 2, 3.5, 3, 2)
draw_edge(ax, 6, 3.5, 5, 2); draw_edge(ax, 6, 3.5, 7, 2)
# Depth 2
for xpos, lbl in [(1,'LL\nd=2'),(3,'LR\nd=2'),(5,'RL\nd=2'),(7,'RR\nd=2')]:
    draw_tree_node(ax, xpos, 2, lbl, '#A9DFBF')  # green = leaves

ax.text(4, 0.8, 'Iteration 1: split depth 0\nIteration 2: split ALL depth-1 nodes\nIteration 3: split ALL depth-2 nodes',
        ha='center', fontsize=8, style='italic',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

# ── Leaf-wise (right) ────────────────────────────────────────────────────────
ax = axes[1]
ax.set_xlim(-1, 9)
ax.set_ylim(-1, 5.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Leaf-Wise Growth (LightGBM)\nAlways split the leaf with the largest gain', fontsize=10)

draw_edge(ax, 4, 5, 2, 3.5); draw_edge(ax, 4, 5, 6, 3.5)
draw_tree_node(ax, 4, 5, 'Root\nd=0', '#AED6F1')
# After split 1: both children exist
draw_tree_node(ax, 2, 3.5, 'L\nd=1', '#F9E79F')  # this leaf has bigger gain
draw_tree_node(ax, 6, 3.5, 'R\nd=1', '#A9DFBF')  # this leaf stays
# Split the LEFT child next (biggest gain)
draw_edge(ax, 2, 3.5, 0.5, 2); draw_edge(ax, 2, 3.5, 3.5, 2)
draw_tree_node(ax, 0.5, 2, 'LL\nd=2', '#F9E79F')  # this now has biggest gain
draw_tree_node(ax, 3.5, 2, 'LR\nd=2', '#A9DFBF')
# Split LL next
draw_edge(ax, 0.5, 2, -0.3, 0.8); draw_edge(ax, 0.5, 2, 1.3, 0.8)
draw_tree_node(ax, -0.3, 0.8, 'LLL\nd=3', '#A9DFBF', radius=0.3, fontsize=7)
draw_tree_node(ax, 1.3,  0.8, 'LLR\nd=3', '#A9DFBF', radius=0.3, fontsize=7)

ax.text(5.5, 1.5, 'Asymmetric!\nRight branch stops\nat depth 1',
        ha='center', fontsize=8, style='italic',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fdebd0', alpha=0.8))
ax.text(4, 0.0, 'Each iteration: find & split THE best leaf\nResult: uneven depth, faster loss reduction',
        ha='center', fontsize=8, style='italic',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Level-Wise vs Leaf-Wise Tree Growth', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 5 — Speed Comparison: LightGBM vs sklearn GradientBoosting

We time both algorithms on our 1500-sample dataset and measure val RMSE. On larger datasets the gap widens dramatically.

**Note:** if LightGBM is not installed, we show sklearn timings only and annotate what would differ.

In [ ]:
# ── Speed comparison ──────────────────────────────────────────────────────────
results = {}

# sklearn GradientBoostingRegressor
t0 = time.time()
sklearn_gb = GradientBoostingRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, random_state=42
)
sklearn_gb.fit(X_train_enc, y_train)
sklearn_time = time.time() - t0
sklearn_rmse = np.sqrt(mean_squared_error(y_val, sklearn_gb.predict(X_val_enc)))
results['sklearn GB'] = {'time': sklearn_time, 'val_rmse': sklearn_rmse}
print(f'sklearn GB  — time: {sklearn_time:.2f}s  |  val RMSE: ${sklearn_rmse:,.0f}')

if LGBM_AVAILABLE:
    # LightGBM — native categorical support
    t0 = time.time()
    lgbm_model = lgb.LGBMRegressor(
        n_estimators=300, num_leaves=31, learning_rate=0.05,
        feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
        random_state=42, verbose=-1
    )
    lgbm_model.fit(
        X_train, y_train,
        categorical_feature=cat_cols,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)]
    )
    lgbm_time = time.time() - t0
    lgbm_rmse = np.sqrt(mean_squared_error(y_val, lgbm_model.predict(X_val)))
    results['LightGBM'] = {'time': lgbm_time, 'val_rmse': lgbm_rmse}
    print(f'LightGBM    — time: {lgbm_time:.2f}s  |  val RMSE: ${lgbm_rmse:,.0f}')
    print(f'\nSpeedup: {sklearn_time/lgbm_time:.1f}× (LightGBM vs sklearn GB)')
else:
    print('\n[LightGBM not available — on this 1500-sample dataset the speedup may be modest.')
    print(' On 100k+ sample datasets, LightGBM is typically 10–20× faster.]')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
names  = list(results.keys())
times  = [results[k]['time']     for k in names]
rmses  = [results[k]['val_rmse'] for k in names]
colors = ['steelblue', 'darkorange'][:len(names)]

axes[0].bar(names, times, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Training Time (seconds)')
axes[0].set_ylabel('Seconds')
for i, v in enumerate(times):
    axes[0].text(i, v + 0.05, f'{v:.2f}s', ha='center', fontweight='bold')

axes[1].bar(names, rmses, color=colors, edgecolor='black', linewidth=0.8)
axes[1].set_title('Validation RMSE ($)')
axes[1].set_ylabel('RMSE ($)')
for i, v in enumerate(rmses):
    axes[1].text(i, v + 200, f'${v:,.0f}', ha='center', fontweight='bold')

plt.suptitle('Speed & Accuracy: LightGBM vs sklearn GB (300 trees)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 6 — Native Categorical Features

LightGBM handles categorical features natively by finding the optimal binary partition of categories at each split — it tries all `2^(k−1) − 1` partitions (or uses a greedy approximation for large `k`). This is better than ordinal encoding which imposes a false ordering.

**How to use:** pass column names via `categorical_feature=` or convert to pandas `Categorical` dtype.

In [ ]:
# ── Native categorical vs one-hot encoding comparison ────────────────────────
if LGBM_AVAILABLE:
    # Version 1: native categorical
    model_native = lgb.LGBMRegressor(
        n_estimators=200, num_leaves=31, learning_rate=0.05,
        random_state=42, verbose=-1
    )
    model_native.fit(X_train, y_train, categorical_feature=cat_cols)
    rmse_native = np.sqrt(mean_squared_error(y_val, model_native.predict(X_val)))

    # Version 2: ordinal encoded
    model_ordinal = lgb.LGBMRegressor(
        n_estimators=200, num_leaves=31, learning_rate=0.05,
        random_state=42, verbose=-1
    )
    model_ordinal.fit(X_train_enc, y_train)
    rmse_ordinal = np.sqrt(mean_squared_error(y_val, model_ordinal.predict(X_val_enc)))

    # Version 3: one-hot encoded
    X_train_ohe = pd.get_dummies(X_train, columns=cat_cols)
    X_val_ohe   = pd.get_dummies(X_val,   columns=cat_cols)
    X_val_ohe   = X_val_ohe.reindex(columns=X_train_ohe.columns, fill_value=0)
    model_ohe = lgb.LGBMRegressor(
        n_estimators=200, num_leaves=31, learning_rate=0.05,
        random_state=42, verbose=-1
    )
    model_ohe.fit(X_train_ohe, y_train)
    rmse_ohe = np.sqrt(mean_squared_error(y_val, model_ohe.predict(X_val_ohe)))

    print('Categorical encoding comparison (LightGBM, 200 trees):')
    print(f'  Native categorical : val RMSE = ${rmse_native:,.0f}')
    print(f'  Ordinal encoded    : val RMSE = ${rmse_ordinal:,.0f}')
    print(f'  One-hot encoded    : val RMSE = ${rmse_ohe:,.0f}')

    cat_results = {
        'Native\nCategorical': rmse_native,
        'Ordinal\nEncoded':    rmse_ordinal,
        'One-Hot\nEncoded':    rmse_ohe
    }
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(cat_results.keys(), cat_results.values(),
                  color=['#2ecc71', '#e74c3c', '#3498db'], edgecolor='black', linewidth=0.8)
    ax.set_title('LightGBM — Categorical Encoding Strategy vs Val RMSE', fontsize=12, fontweight='bold')
    ax.set_ylabel('Validation RMSE ($)')
    for bar, v in zip(bars, cat_results.values()):
        ax.text(bar.get_x() + bar.get_width()/2, v + 100, f'${v:,.0f}',
                ha='center', va='bottom', fontweight='bold')
    ax.set_ylim(0, max(cat_results.values()) * 1.15)
    plt.tight_layout()
    plt.show()
else:
    print('[LightGBM not available — demonstrating concept with sklearn]')
    from sklearn.ensemble import GradientBoostingRegressor as GBR
    model_ord = GBR(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
    model_ord.fit(X_train_enc, y_train)
    rmse_ord = np.sqrt(mean_squared_error(y_val, model_ord.predict(X_val_enc)))
    print(f'sklearn GB with ordinal encoding: val RMSE = ${rmse_ord:,.0f}')
    print('LightGBM native categorical avoids assuming alphabetical/integer ordering')
    print('and finds the optimal binary partition of categories — typically 2-5% better RMSE.')

## Section 7 — Key Hyperparameters: `num_leaves` vs `max_depth`

For leaf-wise growth, `num_leaves` is more meaningful than `max_depth`:
- A tree with `max_depth=5` (level-wise) has at most `2^5 = 32` leaves
- LightGBM's `num_leaves=31` is roughly equivalent in capacity
- But leaf-wise with `num_leaves=31` can form much deeper, unbalanced trees

**Overfitting guard:** `min_data_in_leaf` prevents leaves from splitting when they would have too few samples.

In [ ]:
# ── Hyperparameter scan: num_leaves × min_data_in_leaf ────────────────────────
num_leaves_grid      = [10, 31, 100, 300]
min_data_in_leaf_grid = [1, 10, 50]

heatmap_data = np.zeros((len(min_data_in_leaf_grid), len(num_leaves_grid)))

for i, mdl in enumerate(min_data_in_leaf_grid):
    for j, nl in enumerate(num_leaves_grid):
        if LGBM_AVAILABLE:
            m = lgb.LGBMRegressor(
                n_estimators=200, num_leaves=nl, min_child_samples=mdl,
                learning_rate=0.05, random_state=42, verbose=-1
            )
            m.fit(X_train, y_train, categorical_feature=cat_cols)
            preds = m.predict(X_val)
        else:
            # Approximate with sklearn — max_depth = ceil(log2(num_leaves))
            depth = max(1, int(np.ceil(np.log2(nl))))
            m = GradientBoostingRegressor(
                n_estimators=200, max_depth=depth, min_samples_leaf=mdl,
                learning_rate=0.05, random_state=42
            )
            m.fit(X_train_enc, y_train)
            preds = m.predict(X_val_enc)
        heatmap_data[i, j] = np.sqrt(mean_squared_error(y_val, preds))

heatmap_df = pd.DataFrame(
    heatmap_data,
    index=[f'min_data={v}' for v in min_data_in_leaf_grid],
    columns=[f'num_leaves={v}' for v in num_leaves_grid]
)

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(heatmap_df, annot=True, fmt='.0f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Val RMSE ($)'})
ax.set_title('Validation RMSE ($): num_leaves × min_data_in_leaf\n'
             '(green = lower RMSE = better)', fontsize=11, fontweight='bold')
ax.set_xlabel('num_leaves')
ax.set_ylabel('min_data_in_leaf')
plt.tight_layout()
plt.show()

best_idx = np.unravel_index(heatmap_data.argmin(), heatmap_data.shape)
print(f'\nBest combination:')
print(f'  num_leaves     = {num_leaves_grid[best_idx[1]]}')
print(f'  min_data_in_leaf = {min_data_in_leaf_grid[best_idx[0]]}')
print(f'  val RMSE       = ${heatmap_data[best_idx]:,.0f}')
print('\nNote: large num_leaves + small min_data_in_leaf → high variance (top-right cell)')
print('      small num_leaves → underfitting (left column)')

## Section 8 — Three-Way Comparison: LightGBM vs XGBoost vs sklearn GB

We compare all three on identical tasks: training time, validation RMSE, and approximate memory usage.

In [ ]:
# ── Three-way comparison ──────────────────────────────────────────────────────
comparison = {}

# 1. sklearn GradientBoosting
t0 = time.time()
m_sklearn = GradientBoostingRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, random_state=42
)
m_sklearn.fit(X_train_enc, y_train)
comparison['sklearn GB'] = {
    'time': time.time() - t0,
    'val_rmse': np.sqrt(mean_squared_error(y_val, m_sklearn.predict(X_val_enc))),
    'color': '#3498db'
}

# 2. XGBoost
if XGB_AVAILABLE:
    t0 = time.time()
    m_xgb = xgb.XGBRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        eval_metric='rmse', verbosity=0
    )
    m_xgb.fit(X_train_enc, y_train,
              eval_set=[(X_val_enc, y_val)],
              verbose=False)
    comparison['XGBoost'] = {
        'time': time.time() - t0,
        'val_rmse': np.sqrt(mean_squared_error(y_val, m_xgb.predict(X_val_enc))),
        'color': '#e67e22'
    }
    print('XGBoost included in comparison.')
else:
    print('XGBoost not available — comparison will show sklearn GB and LightGBM only.')

# 3. LightGBM
if LGBM_AVAILABLE:
    t0 = time.time()
    m_lgbm = lgb.LGBMRegressor(
        n_estimators=300, num_leaves=31, learning_rate=0.05,
        feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
        random_state=42, verbose=-1
    )
    m_lgbm.fit(X_train, y_train, categorical_feature=cat_cols)
    comparison['LightGBM'] = {
        'time': time.time() - t0,
        'val_rmse': np.sqrt(mean_squared_error(y_val, m_lgbm.predict(X_val))),
        'color': '#2ecc71'
    }

# Print table
print(f'\n{"Model":<14} {"Train Time (s)":>14} {"Val RMSE ($)":>14}')
print('-' * 45)
for name, info in comparison.items():
    print(f'{name:<14} {info["time"]:>14.2f} {info["val_rmse"]:>14,.0f}')

# Charts
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names  = list(comparison.keys())
colors = [comparison[k]['color'] for k in names]

times = [comparison[k]['time'] for k in names]
axes[0].bar(names, times, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Training Time (300 trees, 1500 samples)')
axes[0].set_ylabel('Seconds')
for i, v in enumerate(times):
    axes[0].text(i, v + 0.01, f'{v:.2f}s', ha='center', fontweight='bold', fontsize=10)

rmses = [comparison[k]['val_rmse'] for k in names]
axes[1].bar(names, rmses, color=colors, edgecolor='black', linewidth=0.8)
axes[1].set_title('Validation RMSE ($)')
axes[1].set_ylabel('RMSE ($)')
for i, v in enumerate(rmses):
    axes[1].text(i, v + 100, f'${v:,.0f}', ha='center', fontweight='bold', fontsize=10)

plt.suptitle('Three-Way Comparison: sklearn GB vs XGBoost vs LightGBM',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 9 — LightGBM vs XGBoost: Reference Comparison Table

In [ ]:
# ── Comparison table rendered as a styled DataFrame ──────────────────────────
table = pd.DataFrame({
    'Aspect':             ['Split finding', 'Tree growth', 'Speed (large data)',
                           'Memory',        'Small dataset', 'Categorical features',
                           'Key hyperparameter', 'Default trees'],
    'LightGBM':           ['Histogram (255 bins)', 'Leaf-wise (best-first)',
                           'Much faster', 'Lower (uint8 bins)', 'Prone to overfit',
                           'Native (optimal partition)', 'num_leaves', '100'],
    'XGBoost':            ['Sorted / Approximate', 'Level-wise',
                           'Slower', 'Higher (float32)', 'More robust',
                           'Must encode', 'max_depth', '100'],
    'sklearn GB':         ['Sorted', 'Level-wise',
                           'Slowest', 'Highest (float64)', 'Most robust',
                           'Must encode', 'max_depth', '100'],
})
table.set_index('Aspect', inplace=True)
table.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('font-weight', 'bold'), ('background-color', '#f0f0f0')]}]
)

## Section 10 — Feature Importance

In [ ]:
# ── Feature importance ────────────────────────────────────────────────────────
if LGBM_AVAILABLE:
    importances = pd.Series(
        m_lgbm.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=True)
else:
    importances = pd.Series(
        m_sklearn.feature_importances_,
        index=X_train_enc.columns
    ).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors_imp = ['#d62728' if v == importances.max() else '#3498db' for v in importances.values]
importances.plot(kind='barh', ax=ax, color=colors_imp, edgecolor='black', linewidth=0.5)
ax.set_title('LightGBM Feature Importance (split-based)', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance score')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## Self-Check Questions

Test your understanding before moving on.

---

**Q1.** LightGBM uses `num_leaves=31` as its default, not `max_depth`. Why is `num_leaves` the more natural parameter for leaf-wise growth?

<details>
<summary>Show answer</summary>

In level-wise growth every tree has exactly `2^max_depth` leaves, so `max_depth` directly controls model complexity. In leaf-wise growth the tree is **asymmetric** — one branch may be 10 levels deep while another is 2 levels deep. Depth has no fixed meaning when branches grow unevenly. `num_leaves`, however, directly caps the total number of leaf nodes regardless of tree shape. Since leaf values are what drive predictions (each leaf stores a constant output), the number of leaves is the true measure of model capacity for leaf-wise trees. Setting `max_depth` in LightGBM is a secondary safeguard, not the primary control.

</details>

---

**Q2.** GOSS keeps all high-gradient samples. Why are high-gradient samples more important than low-gradient ones?

<details>
<summary>Show answer</summary>

The gradient of the loss with respect to the current prediction tells us how wrong the model is on that sample. A **large gradient** means the model is making a big mistake — this sample carries a lot of information for learning the next tree. A **small gradient** means the model already predicts this sample well; it contributes little to identifying the best next split. Formally, the gain of any split is a function of the sum of gradients in each child node. Removing low-gradient samples changes these sums negligibly. Removing high-gradient samples would change them significantly and lead to poor split choices. GOSS exploits this asymmetry: keep all the informative hard cases, subsample the easy ones, and amplify the remaining easy cases by `(1−a)/b` to keep the gain estimates unbiased.

</details>

---

**Q3.** Histogram-based splitting uses 255 bins. This means you lose some precision in split thresholds. Why is this usually acceptable?

<details>
<summary>Show answer</summary>

Three reasons make the precision loss negligible in practice:

1. **Ensemble averaging:** gradient boosting trains hundreds of trees. A suboptimal split threshold in one tree is corrected by subsequent trees. The ensemble does not rely on any single split being exact.
2. **Noise in real data:** real datasets contain measurement error and noise that is typically larger than the quantisation error introduced by 255 bins. Chasing exact split thresholds is fitting noise, not signal.
3. **Theoretical bound (Ke et al. 2017):** the paper proves that the accuracy loss from histogram approximation is bounded by `O(1/num_bins)`. With 255 bins, this bound is very small and empirically the test-set accuracy of LightGBM matches or exceeds exact methods despite the approximation.

The practical gains — 8× memory reduction and 5–20× speedup — far outweigh the tiny accuracy loss.

</details>

## Summary

| Concept | Key Idea | Benefit |
|---|---|---|
| Histogram splitting | Bin features into 255 buckets | 5–20× fewer split evaluations; 8× less memory |
| GOSS | Discard easy samples (small gradient), keep hard ones | 2–4× fewer samples per round with unbiased gain estimates |
| Leaf-wise growth | Always split the leaf with maximum gain | Faster loss reduction per leaf; deeper useful branches |
| `num_leaves` | Primary complexity control for leaf-wise trees | More intuitive than `max_depth` for unbalanced trees |
| `min_data_in_leaf` | Minimum samples per leaf | Main guard against overfitting with leaf-wise growth |
| Native categoricals | Optimal binary partition of categories | Avoids false ordering; typically better than OHE/ordinal |

**When to use LightGBM:**
- Large datasets (100k+ rows) where speed matters
- Datasets with many categorical features
- Kaggle competitions and production tabular ML
- When you want state-of-the-art accuracy with minimal preprocessing

**When to be careful:**
- Small datasets (< 10k rows): high `num_leaves` can overfit aggressively — use `min_data_in_leaf >= 20`
- Always tune `num_leaves` and `min_data_in_leaf` together, not separately

**Next:** Notebook 12 — CatBoost: Ordered Boosting and Native Categoricals Without Target Leakage